In [ ]:
# ================== CLEAN SETUP ==================
import os, warnings, logging, re, random
os.environ["WANDB_DISABLED"] = "true"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ================== IMPORTS ==================
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import nltk
from nltk.corpus import wordnet

from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from torch.nn import CrossEntropyLoss

nltk.download("wordnet")

# ================== TEXT CLEANING ==================
def clean_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

# ================== AUGMENTATION ==================
def synonym_replacement(sentence, n=1):
    words = sentence.split()
    new_words = words.copy()
    random_word_list = list(set([w for w in words if len(wordnet.synsets(w)) > 0]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = wordnet.synsets(random_word)
        if not synonyms:
            continue
        synonym_words = [lemma.name().replace("_", " ") for syn in synonyms for lemma in syn.lemmas() if lemma.name() != random_word]
        if synonym_words:
            synonym = random.choice(synonym_words)
            new_words = [synonym if w == random_word else w for w in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return " ".join(new_words)

def random_deletion(sentence, p=0.1):
    words = sentence.split()
    if len(words) == 1:
        return sentence
    new_words = [w for w in words if random.uniform(0, 1) > p]
    if not new_words:
        return random.choice(words)
    return " ".join(new_words)

def augment_text(text):
    if random.random() < 0.5:
        return synonym_replacement(text, n=1)
    else:
        return random_deletion(text, p=0.1)

# ================== LOAD DATA ==================
df = pd.read_csv("Dataset_1_2_3_4.csv")

if df["label"].dtype == "object":
    label_mapping = {"Non-Hate": 0, "Hate": 1}
    df["label"] = df["label"].map(label_mapping)

df["label"] = df["label"].astype(int)
df["text"] = df["text"].astype(str).apply(clean_text) # Added .astype(str) here

# Augment training data
augmented_rows = []
for text, label in zip(df["text"], df["label"]):
    if random.random() < 0.3:
        augmented_text = augment_text(text)
        augmented_rows.append({"text": augmented_text, "label": label})

df_aug = pd.DataFrame(augmented_rows)
df = pd.concat([df, df_aug]).reset_index(drop=True)

# Split dataset
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# ================== CLASS WEIGHTS ==================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# ================== TOKENIZER ==================
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

class TextDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=tokenizer, max_len=192):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TextDataset(train_texts, train_labels)
test_dataset  = TextDataset(test_texts, test_labels)

# ================== MODEL ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights.to(device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# ================== TRAINING ARGS ==================
training_args = TrainingArguments(
    output_dir="./results_roberta",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,   # 🔹 changed to 10
    weight_decay=0.01,
    warmup_steps=300,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# ================== TRAIN ==================
trainer.train()

# ================== EVALUATE ==================
predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(axis=-1)

print("\n✅ Test Accuracy:", accuracy_score(test_labels, y_pred))
print("\n✅ Classification Report:\n", classification_report(test_labels, y_pred, target_names=["Non-Hate", "Hate"]))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Using device: cuda


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

{'loss': 0.612, 'grad_norm': 11.951534271240234, 'learning_rate': 1.86e-05, 'epoch': 1.0}
{'eval_loss': 0.5589413642883301, 'eval_accuracy': 0.717731129968736, 'eval_runtime': 22.9147, 'eval_samples_per_second': 97.71, 'eval_steps_per_second': 3.055, 'epoch': 1.0}
{'loss': 0.4542, 'grad_norm': 12.580354690551758, 'learning_rate': 1.7928000000000002e-05, 'epoch': 2.0}
{'eval_loss': 0.45846572518348694, 'eval_accuracy': 0.7896382313532827, 'eval_runtime': 22.9346, 'eval_samples_per_second': 97.625, 'eval_steps_per_second': 3.052, 'epoch': 2.0}
{'loss': 0.3052, 'grad_norm': 14.432851791381836, 'learning_rate': 1.5688e-05, 'epoch': 3.0}
{'eval_loss': 0.5106534957885742, 'eval_accuracy': 0.8222420723537294, 'eval_runtime': 22.9227, 'eval_samples_per_second': 97.676, 'eval_steps_per_second': 3.054, 'epoch': 3.0}
{'loss': 0.1905, 'grad_norm': 11.43935489654541, 'learning_rate': 1.3448e-05, 'epoch': 4.0}
{'eval_loss': 0.5069689750671387, 'eval_accuracy': 0.8450200982581509, 'eval_runtime': 22.